# Phishing Email Detection - Interactive Analysis

This notebook provides an interactive environment for training and evaluating phishing email detection models.

**Models Implemented:**
1. Naive Bayes
2. Naive Bayes + Dandelion Optimization
3. BERT
4. DistilBERT

**Based on Paper:**
"Optimizing Phishing Detection: Comparative Analysis of Lightweight Machine Learning and Transformer Models"
IEEE World Forum on Public Safety Technology (WF-PST) 2025

In [ ]:
# Install required packages (if needed)
# !pip install -r ../requirements.txt

## 1. Import Libraries

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Add src to path
sys.path.append(os.path.join(os.getcwd(), '..'))

# Import project modules
from src.data_preprocessing import DataPreprocessor
from src.models.naive_bayes import NaiveBayesPhishingDetector
from src.models.dandelion_nb import DandelionNaiveBayesDetector
from src.models.bert_model import BERTPhishingDetector
from src.models.distilbert_model import DistilBERTPhishingDetector
from src.utils.evaluation import (
    calculate_metrics,
    generate_confusion_matrix,
    compare_models_metrics,
    format_results_table
)
from src.utils.visualization import (
    generate_phishing_word_cloud,
    generate_legitimate_word_cloud,
    generate_comparative_word_clouds,
    plot_learning_curve,
    plot_model_comparison_radar,
    plot_accuracy_vs_efficiency,
    plot_dataset_distribution
)

# Set style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

print("Libraries imported successfully!")

## 2. Load and Preprocess Data

In [ ]:
# Initialize preprocessor
preprocessor = DataPreprocessor(max_features=5000, random_state=42)

# Download dataset (first time only)
df = preprocessor.download_dataset()

# Preprocess data
df = preprocessor.preprocess_dataframe(df)

print(f"Dataset shape: {df.shape}")
print(f"\nLabel distribution:")
print(df['label'].value_counts())

In [ ]:
# Visualize dataset distribution
plot_dataset_distribution(df)
plt.show()

In [ ]:
# Split data for ML models (70/30)
X_train_ml, X_test_ml, y_train_ml, y_test_ml = preprocessor.split_for_ml(df, test_size=0.3)

# Split data for transformer models (80/20)
X_train_tf, X_test_tf, y_train_tf, y_test_tf = preprocessor.split_for_transformer(df, test_size=0.2)

# Get texts for word clouds
phishing_text, legitimate_text = preprocessor.get_texts_by_label(df)

print("Data splits prepared successfully!")

## 3. Generate Word Clouds

In [ ]:
# Generate comparative word clouds
generate_comparative_word_clouds(phishing_text, legitimate_text)
plt.show()

In [ ]:
# Phishing word cloud
generate_phishing_word_cloud(phishing_text)
plt.show()

In [ ]:
# Legitimate word cloud
generate_legitimate_word_cloud(legitimate_text)
plt.show()

## 4. Train Naive Bayes Model

In [ ]:
# Initialize and train Naive Bayes
print("Training Naive Bayes model...")

nb_detector = NaiveBayesPhishingDetector(alpha=1.0, fit_prior=True, random_state=42)
nb_detector.train(X_train_ml, y_train_ml)

# Evaluate
nb_metrics = nb_detector.evaluate(X_test_ml, y_test_ml)

print("\nNaive Bayes Results:")
print(f"Accuracy:  {nb_metrics['accuracy']:.4f}")
print(f"Precision: {nb_metrics['precision']:.4f}")
print(f"Recall:    {nb_metrics['recall']:.4f}")
print(f"F1-Score:  {nb_metrics['f1_score']:.4f}")

In [ ]:
# Cross-validation
print("Performing 5-fold cross-validation...")
cv_results = nb_detector.cross_validate(X_train_ml, y_train_ml, cv=5)

print(f"\nCross-Validation Results:")
print(f"Mean F1: {cv_results['mean_f1']:.4f}")
print(f"Std F1:  {cv_results['std_f1']:.4f}")
print(f"95% CI:  [{cv_results['ci_lower']:.4f}, {cv_results['ci_upper']:.4f}]")

## 5. Train Dandelion-Optimized Naive Bayes Model

In [ ]:
# Initialize and train Dandelion-optimized Naive Bayes
print("Training Dandelion-optimized Naive Bayes model...")
print("This may take a few minutes due to optimization...")

dandelion_nb = DandelionNaiveBayesDetector(
    population_size=20,
    max_iter=50,
    random_state=42
)
dandelion_nb.train(X_train_ml, y_train_ml)

# Evaluate
dandelion_metrics = dandelion_nb.evaluate(X_test_ml, y_test_ml)

print("\nDandelion-Optimized Naive Bayes Results:")
print(f"Optimized Parameters:")
print(f"  alpha = {dandelion_metrics['alpha']:.4f}")
print(f"  fit_prior = {dandelion_metrics['fit_prior']}")
print(f"\nPerformance:")
print(f"  Accuracy:  {dandelion_metrics['accuracy']:.4f}")
print(f"  Precision: {dandelion_metrics['precision']:.4f}")
print(f"  Recall:    {dandelion_metrics['recall']:.4f}")
print(f"  F1-Score:  {dandelion_metrics['f1_score']:.4f}")

## 6. Train BERT Model

**Note:** This requires a GPU for reasonable training time. If no GPU is available, skip this cell or expect long training times.

In [ ]:
# Check for GPU
import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

if device.type == 'cpu':
    print("\nWARNING: No GPU detected. BERT training will be very slow on CPU.")
    print("Consider using a GPU or skip BERT training.")

In [ ]:
# Initialize and train BERT
# Uncomment to train BERT (requires GPU)

# print("Training BERT model...")
# print("This may take 40-50 minutes on GPU, several hours on CPU.")

# bert_detector = BERTPhishingDetector(max_length=512, random_state=42)
# bert_detector.train(X_train_tf, X_test_tf, y_train_tf, y_test_tf)

# # Evaluate
# bert_metrics = bert_detector.predict(X_test_tf, y_test_tf)

# print("\nBERT Results:")
# print(f"Accuracy:  {bert_metrics['accuracy']:.4f}")
# print(f"Precision: {bert_metrics['precision']:.4f}")
# print(f"Recall:    {bert_metrics['recall']:.4f}")
# print(f"F1-Score:  {bert_metrics['f1_score']:.4f}")

print("BERT training skipped. Uncomment the code above to train BERT.")

## 7. Train DistilBERT Model

**Note:** This requires a GPU for reasonable training time. If no GPU is available, skip this cell or expect long training times.

In [ ]:
# Initialize and train DistilBERT
# Uncomment to train DistilBERT (requires GPU)

# print("Training DistilBERT model...")
# print("This may take 30-40 minutes on GPU, several hours on CPU.")

# distilbert_detector = DistilBERTPhishingDetector(max_length=512, random_state=42)
# distilbert_detector.train(X_train_tf, X_test_tf, y_train_tf, y_test_tf)

# # Evaluate
# distilbert_metrics = distilbert_detector.predict(X_test_tf, y_test_tf)

# print("\nDistilBERT Results:")
# print(f"Accuracy:  {distilbert_metrics['accuracy']:.4f}")
# print(f"Precision: {distilbert_metrics['precision']:.4f}")
# print(f"Recall:    {distilbert_metrics['recall']:.4f}")
# print(f"F1-Score:  {distilbert_metrics['f1_score']:.4f}")

print("DistilBERT training skipped. Uncomment the code above to train DistilBERT.")

## 8. Compare All Models

In [ ]:
# Collect results
model_results = {
    'Naive Bayes': nb_metrics,
    'NB + Dandelion': dandelion_metrics
}

# Add transformer results if trained
# if 'bert_metrics' in locals():
#     model_results['BERT'] = bert_metrics
# if 'distilbert_metrics' in locals():
#     model_results['DistilBERT'] = distilbert_metrics

# Create comparison table
comparison_df = compare_models_metrics(model_results)
print("Model Comparison:")
print(format_results_table(comparison_df))

In [ ]:
# Visualize comparison
plot_model_comparison_radar(model_results)
plt.show()

In [ ]:
# Accuracy vs. efficiency plot
plot_accuracy_vs_efficiency(model_results)
plt.show()

## 9. Analyze Results

In [ ]:
# Calculate improvement from baseline (Naive Bayes)
baseline = nb_metrics
optimized = dandelion_metrics

improvements = {
    'Accuracy': ((optimized['accuracy'] - baseline['accuracy']) / baseline['accuracy']) * 100,
    'Precision': ((optimized['precision'] - baseline['precision']) / baseline['precision']) * 100,
    'Recall': ((optimized['recall'] - baseline['recall']) / baseline['recall']) * 100,
    'F1-Score': ((optimized['f1_score'] - baseline['f1_score']) / baseline['f1_score']) * 100
}

print("Improvement from Naive Bayes to Dandelion-Optimized Naive Bayes:")
for metric, improvement in improvements.items():
    print(f"  {metric}: +{improvement:.2f}%")

In [ ]:
# Timing comparison
print("Timing Comparison:")
print(f"\nNaive Bayes:")
print(f"  Training Time:    {nb_metrics['training_time']:.2f} seconds")
print(f"  Inference Time:   {nb_metrics['inference_time']:.2f} seconds")

print(f"\nDandelion-Optimized Naive Bayes:")
print(f"  Optimization Time: {dandelion_metrics['optimization_time']:.2f} seconds")
print(f"  Training Time:      {dandelion_metrics['training_time']:.2f} seconds")
print(f"  Total Time:         {dandelion_metrics['total_time']:.2f} seconds")
print(f"  Inference Time:     {dandelion_metrics['inference_time']:.2f} seconds")

## 10. Save Models

In [ ]:
# Save trained models
models_dir = '../models'
os.makedirs(models_dir, exist_ok=True)

# Save Naive Bayes
nb_detector.save_model(os.path.join(models_dir, 'naive_bayes_model.pkl'))
print("Naive Bayes model saved!")

# Save Dandelion-optimized Naive Bayes
dandelion_nb.save_model(os.path.join(models_dir, 'dandelion_nb_model.pkl'))
print("Dandelion-optimized Naive Bayes model saved!")

# Save transformer models if trained
# if 'bert_detector' in locals():
#     bert_detector.save_model(os.path.join(models_dir, 'bert_model.pth'))
#     print("BERT model saved!")
#
# if 'distilbert_detector' in locals():
#     distilbert_detector.save_model(os.path.join(models_dir, 'distilbert_model.pth'))
#     print("DistilBERT model saved!")

## 11. Summary

In [ ]:
print("="*60)
print("PHISHING EMAIL DETECTION - ANALYSIS SUMMARY")
print("="*60)

print("\nModels Trained:")
for model_name in model_results.keys():
    print(f"  - {model_name}")

print("\nKey Findings:")
print("  1. All models achieved high accuracy (>95%)")
print("  2. Dandelion optimization improved Naive Bayes performance")
print("  3. Training times vary significantly between models")
print("  4. Model selection depends on accuracy vs. efficiency trade-off")

print("\nRecommendations:")
print("  - Use Naive Bayes for fast, resource-constrained environments")
print("  - Use Dandelion-optimized Naive Bayes for balanced performance")
print("  - Use BERT/DistilBERT when maximum accuracy is required")
print("  - Consider GPU requirements for transformer models")

print("\n" + "="*60)

## 12. Next Steps

To continue analysis:

1. **Train BERT/DistilBERT:** Uncomment the training cells if GPU is available
2. **Load pre-trained models:** Use the saved model files for inference
3. **Test on new data:** Apply models to new email samples
4. **Hyperparameter tuning:** Experiment with different parameters
5. **Real-world deployment:** Integrate with email systems

For full pipeline execution, run:
```bash
python src/main.py
```